In [43]:
from pyspark.sql import SparkSession

In [44]:
spark = SparkSession.builder.appName("DemoSparkApp").getOrCreate()

In [45]:
df1 = spark.read.csv("data/customers.csv")

In [46]:
# Customers
df1 = spark.read.csv(
    "data/customers.csv",
    header=True,
    inferSchema=True
)

# Order Items
df2 = spark.read.csv(
    "data/order_items.csv",
    header=True,
    inferSchema=True
)

# Orders
df3 = spark.read.csv(
    "data/orders.csv",
    header=True,
    inferSchema=True
)

# Products
df4 = spark.read.csv(
    "data/products.csv",
    header=True,
    inferSchema=True
)

# Returns
df5 = spark.read.csv(
    "data/returns.csv",
    header=True,
    inferSchema=True
)

In [47]:
result1 = df1.count()
result2 = df2.count()
result3 = df3.count()
result4 = df4.count()
result5 = df5.count()

In [48]:
print("total customer: ",result1)
print("total order_items: ",result2)
print("total orders: ",result3)
print("total products: ",result4)
print("total returns: ",result5)

total customer:  100000
total order_items:  3000000
total orders:  1000000
total products:  50000
total returns:  100000


In [49]:


result = spark.sql("""
SELECT
    category,
    SUM(unit_cost) AS total_sales
FROM df4
GROUP BY category
""")

result.show()
result.write.mode("overwrite").csv("output/df4.csv", header=True)

+--------------+------------------+
|      category|       total_sales|
+--------------+------------------+
|Home & Kitchen| 2901364.330000004|
|        Sports| 2853163.040000003|
|   Electronics|2864604.7399999946|
|      Clothing| 2841424.610000002|
|         Books|2853871.8500000075|
|        Beauty|2919388.7500000037|
|          Toys|2851913.1100000013|
+--------------+------------------+



In [52]:
df1.createOrReplaceTempView("df1")
df2.createOrReplaceTempView("df2")
df3.createOrReplaceTempView("df3")
df4.createOrReplaceTempView("df4")

result = spark.sql("""
SELECT
    c.customer_id,
    c.customer_name,
    SUM(oi.quantity * p.unit_cost) AS total_purchase
FROM df1 c
JOIN df3 o
ON c.customer_id = o.customer_id
JOIN df2 oi
ON o.order_id = oi.order_id
JOIN df4 p
ON oi.product_id = p.product_id
GROUP BY c.customer_id, c.customer_name
ORDER BY total_purchase DESC
LIMIT 10
""")

result.show()

[Stage 148:>                                                        (0 + 2) / 2]

+-----------+--------------+------------------+
|customer_id| customer_name|    total_purchase|
+-----------+--------------+------------------+
|      93094|Customer_93094|          122887.1|
|      64560|Customer_64560|116660.08000000002|
|      23289|Customer_23289|109359.41999999998|
|      61218|Customer_61218|107647.11000000003|
|      40442|Customer_40442|104905.09000000001|
|      52275|Customer_52275|103149.83000000003|
|      52034|Customer_52034|102637.66999999998|
|      45631|Customer_45631|         102636.03|
|      84830|Customer_84830|102471.69999999998|
|      82593|Customer_82593|101936.83999999997|
+-----------+--------------+------------------+

